## GH-25 Filters

In [ ]:
import orjson
from tqdm import tqdm

INPUT = "datasets/gh_25_github_io_repos_collapsed.jsonl"
OUTPUT = "datasets/gh_25_github_io_repos_filtered.jsonl"

USEFUL_EXTS = {
    ".html", ".css", ".js", ".ts", ".tsx", ".jsx",
    ".vue", ".astro", ".scss", ".sass", ".less", ".py"
}

NON_USEFUL_EXTS = {
    ".md", ".rst", ".txt"
}

CONFIG_EXTS = {
    ".yml", ".yaml", ".json", ".toml"
}

MIN_TOTAL_FILES = 10
MIN_TOTAL_SIZE = 100_000
MIN_USEFUL_FRACTION = 0.7
MAX_NON_USEFUL_FRACTION = 0.2


def is_good_web_repo(obj):
    paths = obj["file_path"]
    sizes = obj["size"]

    total_files = len(paths)
    total_size = sum(sizes)

    if total_files < MIN_TOTAL_FILES:
        return False
    if total_size < MIN_TOTAL_SIZE:
        return False

    useful = non_useful = config = 0

    for p in paths:
        p = p.lower()
        if any(p.endswith(ext) for ext in USEFUL_EXTS):
            useful += 1
        elif any(p.endswith(ext) for ext in NON_USEFUL_EXTS):
            non_useful += 1
        elif any(p.endswith(ext) for ext in CONFIG_EXTS):
            config += 1

    useful_frac = useful / total_files
    non_useful_frac = non_useful / total_files

    if non_useful_frac > MAX_NON_USEFUL_FRACTION:
        return False

    return useful_frac >= MIN_USEFUL_FRACTION


kept = 0
with open(INPUT, "rb") as fin, open(OUTPUT, "wb") as fout:
    for line in tqdm(fin, desc="Filtering repos"):
        obj = orjson.loads(line)
        if is_good_web_repo(obj):
            fout.write(orjson.dumps(obj) + b"\n")
            kept += 1

print(f"✅ Kept {kept} high-quality web repos")

: 

## Stack Filters

In [ ]:
USEFUL_WEB_LANGS = {
    "HTML", "CSS", "JavaScript", "TypeScript",
    "TSX", "JSX", "Vue", "Astro", "SCSS", "Sass", "Less"
}

NON_USEFUL_LANGS = {
    "Markdown", "RMarkdown", "Text"
}

MIN_TOTAL_FILES = 10
MIN_TOTAL_SIZE = 100_000          # bytes proxy
MIN_USEFUL_FRACTION = 0.7
MAX_NON_USEFUL_FRACTION = 0.2

def is_good_web_repo(row):
    files = row.get("files")
    if not files:
        return False

    total_files = len(files)
    if total_files < MIN_TOTAL_FILES:
        return False

    total_size = 0
    useful = 0
    non_useful = 0

    for f in files:
        total_size += f.get("length_bytes", 0)

        lang = f.get("language")
        path = f.get("path", "").lower()

        if lang in USEFUL_WEB_LANGS:
            useful += 1
        elif lang in NON_USEFUL_LANGS or path.endswith((".md", ".rst", ".txt")):
            non_useful += 1

    if total_size < MIN_TOTAL_SIZE:
        return False

    useful_frac = useful / total_files
    non_useful_frac = non_useful / total_files

    if non_useful_frac > MAX_NON_USEFUL_FRACTION:
        return False

    return useful_frac >= MIN_USEFUL_FRACTION

from datasets import load_dataset
from tqdm import tqdm
import json
import multiprocessing as mp

OUTPUT_PATH = "github_io_websites_filtered.jsonl"

# ---- Load dataset ----
ds = load_dataset(
    "Ayush-Singh/stack-filtered-githubio",
    split="train",
    streaming=False,
    num_proc=4
)

print(f"📦 Loaded {len(ds)} repos")

# ---- Parallel filtering ----
filtered = ds.filter(
    is_good_web_repo,
    num_proc=mp.cpu_count(),
    desc="Filtering web-quality GitHub.io repos"
)

print(f"✅ After filtering: {len(filtered)} repos")

# ---- Write JSONL + count lines ----
num_lines = 0
with open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for row in tqdm(filtered, desc="Writing JSONL"):
        json.dump(
            {
                "repo_name": row["repo_name"],
                "repo_url": row["repo_url"],
                "num_files": row["num_files"],
                "files": [
                    {
                        "path": f["path"],
                        "language": f.get("language"),
                        "length_bytes": f.get("length_bytes", 0),
                    }
                    for f in row["files"]
                ],
            },
            fout
        )
        fout.write("\n")
        num_lines += 1

print(f"📁 Saved to {OUTPUT_PATH}")
print(f"📊 JSONL lines written: {num_lines}")